In [ ]:
"""
Customer360 Navigator Enterprise Suite - BP1 Model Persistence notebook (Hardening Step 2).
Single consolidated code cell (platform convention). Idempotent - safe to re-run.

Not a numbered gate (Gates 1-6 plus the executive-rollup "Gate 7" are the Master Plan's fixed
governance cycle - PROJECT_STRUCTURE_LOCKED.md does not permit inventing a new gate number).
This is a standalone productization step, run only after BP1 Gate 5 (Decision Layer & Reporting)
is real-run confirmed: it refits the already-confirmed champion on the identical full-train split
Gate 5 already validated, persists it to models/bp1_customer_intent_classification/ as a joblib
bundle (TF-IDF + classifier pipeline plus the label encoder needed to map predictions back to
real intent-label strings), and proves the persisted artifact is a faithful copy by reloading it
and reproducing Gate 5's exact recorded accuracy.

Reuses src/models/bp1_intent_classifier.py (Gate 6's own extraction) for pipeline construction and
src/models/model_persistence.py (new, this step) for the actual save/load/predict mechanics -
HYPER: no pipeline-construction or joblib-boilerplate code is duplicated inline a further time.
"""

import os, sys, json, time, platform, warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
def _find_project_root() -> Path:
    marker = "PROJECT_STRUCTURE_LOCKED.md"
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        if (Path(env_override) / marker).exists():
            return Path(env_override)
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {env_override!r} but {marker} was not found there. "
            "Fix the environment variable rather than removing this check."
        )
    start = Path.cwd()
    cur = start
    for _ in range(8):
        if (cur / marker).exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker in filenames:
            return Path(depth_root)
    raise RuntimeError(
        f"Could not resolve PROJECT_ROOT: no {marker} found by walking up from {start}, nor by "
        "searching up to 3 levels below it. Fix: add a cell at the TOP of this notebook (before "
        "this cell runs) with:\n"
        '    import os; os.environ["C360_PROJECT_ROOT"] = r"C:\\Users\\rnand\\Documents\\'
        'Customer360_Navigator_Enterprise_Suite"\n'
        "then re-run from the top."
    )

PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP performance configuration - FIRST, before any heavy import
# ============================================================
from utils.performance_setup import (  # noqa: E402
    assert_within_ram_ceiling,
    configure_performance,
    load_resource_limits,
)

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)
RESOURCE_LIMITS = load_resource_limits(PROJECT_ROOT)
assert_within_ram_ceiling(RESOURCE_LIMITS)

# ============================================================
# SECTION 3: Heavy imports + flush-forcing print override
# ============================================================
import builtins  # noqa: E402
import functools  # noqa: E402
import importlib.metadata  # noqa: E402

import pandas as pd  # noqa: E402
import sklearn  # noqa: E402
import yaml  # noqa: E402
from sklearn.metrics import accuracy_score  # noqa: E402
from sklearn.preprocessing import LabelEncoder  # noqa: E402

print = functools.partial(builtins.print, flush=True)

from models.bp1_intent_classifier import make_candidates, make_pipeline  # noqa: E402
from models.model_persistence import load_model_bundle, predict_bp1, save_model_bundle  # noqa: E402

CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_EXTERNAL_DIR = PROJECT_ROOT / "data" / "external"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp1_customer_intent_classification" / "artifacts"
MODELS_DIR = PROJECT_ROOT / "models" / "bp1_customer_intent_classification"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# SECTION 4: Load Gate 3/4/5's real results - champion read LIVE, never hardcoded. Requires
# Gate 5 (not just Gate 4) to already be real-run confirmed, since this step persists the exact
# fitted state Gate 5 already validated end to end (same full-train refit, same accuracy).
# ============================================================
bp1_config_path = CONFIGS_DIR / "bp1_customer_intent_classification.yaml"
with open(bp1_config_path, "r", encoding="utf-8") as f:
    bp1_config = yaml.safe_load(f)
target_def = bp1_config.get("target_definition")
assert target_def is not None, "[CHECK FAILED] target_definition is null - run BP1 Gate 1 first."
gate3_block = bp1_config.get("gate3_model_benchmark")
assert gate3_block is not None, "[CHECK FAILED] gate3_model_benchmark missing - run BP1 Gate 3 first."
gate5_block = bp1_config.get("gate5_decision_layer")
assert gate5_block is not None, (
    "[CHECK FAILED] gate5_decision_layer missing from configs/bp1_customer_intent_classification.yaml - "
    "run BP1 Gate 5 (Decision Layer & Reporting) for real before this model-persistence step."
)
PRIMARY_TARGET = target_def["primary_target"]
FEATURE_COL = target_def["feature_variable"]

gate4_json_path = ARTIFACTS_DIR / "gate4_statistical_validation.json"
assert gate4_json_path.exists(), f"[CHECK FAILED] {gate4_json_path} not found - run BP1 Gate 4 first."
with open(gate4_json_path, "r", encoding="utf-8") as f:
    gate4_results = json.load(f)
CHAMPION_NAME = gate4_results["champion_model"]
assert CHAMPION_NAME == gate3_block["champion_model"], (
    f"[CHECK FAILED] Champion mismatch: Gate 4 recorded '{CHAMPION_NAME}' but Gate 3's config block says "
    f"'{gate3_block['champion_model']}' - these must agree; re-run Gate 3/4."
)

gate5_json_path = ARTIFACTS_DIR / "gate5_decision_layer_summary.json"
assert gate5_json_path.exists(), f"[CHECK FAILED] {gate5_json_path} not found - run BP1 Gate 5 first."
with open(gate5_json_path, "r", encoding="utf-8") as f:
    gate5_summary = json.load(f)
assert gate5_summary["champion_model"] == CHAMPION_NAME, (
    f"[CHECK FAILED] Gate 5 recorded champion '{gate5_summary['champion_model']}' does not match "
    f"Gate 4's '{CHAMPION_NAME}' - re-run Gate 5."
)
gate5_recorded_accuracy = float(gate5_summary["overall_test_accuracy_recomputed"])
gate3_recorded_test_accuracy = float(gate3_block["held_out_test_accuracy"])
print(f"[OK] Champion (live, re-verified against Gate 3 + Gate 4 + Gate 5): {CHAMPION_NAME}")

# ============================================================
# SECTION 5: Load BANKING77 train/test - identical loading to Gates 3/4/5
# ============================================================
B77_TRAIN_PATH = DATA_EXTERNAL_DIR / "banking77_train.csv"
B77_TEST_PATH = DATA_EXTERNAL_DIR / "banking77_test.csv"
train_df = pd.read_csv(B77_TRAIN_PATH, dtype={"text": str, "category": str})
test_df = pd.read_csv(B77_TEST_PATH, dtype={"text": str, "category": str})

label_encoder = LabelEncoder().fit(train_df[PRIMARY_TARGET])
X_train, y_train = train_df[FEATURE_COL], label_encoder.transform(train_df[PRIMARY_TARGET])
X_test, y_test_labels = test_df[FEATURE_COL], test_df[PRIMARY_TARGET]
y_test = label_encoder.transform(y_test_labels)
N_CLASSES = len(label_encoder.classes_)
print(f"[OK] Re-loaded BANKING77 (train={len(X_train):,}, test={len(X_test):,}, {N_CLASSES} classes).")

# ============================================================
# SECTION 6: Refit champion on FULL train - reuses src/models/bp1_intent_classifier.py's shared
# pipeline-construction functions (HYPER), the identical hyperparameters/random_state Gates 3/4/5
# already used, never re-defined inline here.
# ============================================================
assert_within_ram_ceiling(RESOURCE_LIMITS)
candidates = make_candidates(random_state=bp1_config["random_state"])
champion_pipeline = make_pipeline(CHAMPION_NAME, candidates)

print(f"\n[PERSIST] Refitting champion ({CHAMPION_NAME}) on the full train split...")
t0 = time.perf_counter()
champion_pipeline.fit(X_train, y_train)
fit_seconds = time.perf_counter() - t0
print(f"[PERSIST] Fit done in {fit_seconds:.1f}s")

y_pred_encoded = champion_pipeline.predict(X_test)
fresh_accuracy = float(accuracy_score(y_test, y_pred_encoded))
diff_vs_gate3 = abs(fresh_accuracy - gate3_recorded_test_accuracy)
diff_vs_gate5 = abs(fresh_accuracy - gate5_recorded_accuracy)
print(f"[CHECK] Fresh refit test accuracy: {fresh_accuracy:.6f} "
      f"(Gate 3 recorded: {gate3_recorded_test_accuracy:.6f}, diff={diff_vs_gate3:.6f}; "
      f"Gate 5 recorded: {gate5_recorded_accuracy:.6f}, diff={diff_vs_gate5:.6f})")

# ============================================================
# SECTION 7: Assemble and persist the model bundle (src/models/model_persistence.py)
# ============================================================
generated_at = datetime.now(timezone.utc).isoformat()
bundle = {
    "bp_id": "bp1",
    "champion_name": CHAMPION_NAME,
    "pipeline": champion_pipeline,
    "label_encoder": label_encoder,
    "class_names": list(label_encoder.classes_),
    "metadata": {
        "gate3_recorded_test_accuracy": gate3_recorded_test_accuracy,
        "gate5_recorded_test_accuracy": gate5_recorded_accuracy,
        "fresh_refit_test_accuracy": fresh_accuracy,
        "n_train_rows": int(len(X_train)),
        "n_test_rows": int(len(X_test)),
        "n_classes": int(N_CLASSES),
        "random_state": bp1_config["random_state"],
        "fit_seconds": round(fit_seconds, 2),
        "python_version": platform.python_version(),
        "sklearn_version": sklearn.__version__,
        "joblib_version": importlib.metadata.version("joblib"),
        "generated_at_utc": generated_at,
    },
}

out_path = MODELS_DIR / "bp1_champion_pipeline.joblib"
save_stats = save_model_bundle(bundle, out_path)
print(f"\n[SAVED] {out_path.relative_to(PROJECT_ROOT)} "
      f"({save_stats['size_bytes']:,} bytes, sha256={save_stats['sha256'][:16]}...)")

# ============================================================
# SECTION 8: Reload the persisted bundle and prove it is a faithful, usable copy - not merely
# that the file exists. Predicts via predict_bp1() (the same function any future inference
# service will call), never re-touching the in-memory champion_pipeline object.
# ============================================================
reloaded_bundle = load_model_bundle(out_path)
reload_result = predict_bp1(reloaded_bundle, X_test.tolist())
reloaded_pred_labels = reload_result["predicted_label"]
reloaded_accuracy = float(accuracy_score(y_test_labels.tolist(), reloaded_pred_labels))
reload_accuracy_diff = abs(reloaded_accuracy - fresh_accuracy)
print(f"[CHECK] Reloaded-bundle accuracy via predict_bp1(): {reloaded_accuracy:.6f} "
      f"(fresh in-memory: {fresh_accuracy:.6f}, diff={reload_accuracy_diff:.10f})")

# ============================================================
# SECTION 9: Write a human-readable metadata sidecar (the .joblib itself is gitignored per this
# project's own .gitignore - models/**/*.joblib - so this JSON is the artifact's committed,
# version-controlled record of what was persisted and how it was verified).
# ============================================================
metadata_record = {
    "bp_id": "bp1",
    "champion_model": CHAMPION_NAME,
    "joblib_relative_path": str(out_path.relative_to(PROJECT_ROOT)),
    "joblib_size_bytes": save_stats["size_bytes"],
    "joblib_sha256": save_stats["sha256"],
    "fresh_refit_test_accuracy": round(fresh_accuracy, 6),
    "gate3_recorded_test_accuracy": round(gate3_recorded_test_accuracy, 6),
    "gate5_recorded_test_accuracy": round(gate5_recorded_accuracy, 6),
    "accuracy_diff_vs_gate3": round(diff_vs_gate3, 6),
    "accuracy_diff_vs_gate5": round(diff_vs_gate5, 6),
    "reload_verified_accuracy": round(reloaded_accuracy, 6),
    "reload_accuracy_diff": round(reload_accuracy_diff, 10),
    "bundle_keys": sorted(bundle.keys()),
    "python_version": platform.python_version(),
    "sklearn_version": sklearn.__version__,
    "joblib_version": importlib.metadata.version("joblib"),
    "generated_at_utc": generated_at,
}
metadata_path = MODELS_DIR / "bp1_model_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata_record, f, indent=2)
print(f"[SAVED] {metadata_path.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 10: Update the established model_inventory_entry.json (idempotent, same pattern every
# other gate already uses) and the BP config YAML's marker-delimited block.
# ============================================================
inventory_path = ARTIFACTS_DIR / "model_inventory_entry.json"
with open(inventory_path, "r", encoding="utf-8") as f:
    model_inventory_entry = json.load(f)
model_inventory_entry["status"] = "Model persistence (Hardening Step 2) complete"
model_inventory_entry["model_persistence_joblib_path"] = str(out_path.relative_to(PROJECT_ROOT))
model_inventory_entry["model_persistence_sha256"] = save_stats["sha256"]
model_inventory_entry["model_persistence_reload_verified"] = reload_accuracy_diff < 1e-9
model_inventory_entry["model_persistence_generated_at_utc"] = generated_at
with open(inventory_path, "w", encoding="utf-8") as f:
    json.dump(model_inventory_entry, f, indent=2)
print(f"[SAVED] {inventory_path.relative_to(PROJECT_ROOT)} (model_persistence fields added)")

from utils.bp1_config_sync import write_gate_block  # noqa: E402

persistence_marker = "# --- Model Persistence (Hardening Step 2) results (appended, idempotent overwrite) ---"
persistence_block_lines = [
    "model_persistence:",
    f'  champion_model: "{CHAMPION_NAME}"',
    f'  joblib_relative_path: "{out_path.relative_to(PROJECT_ROOT).as_posix()}"',
    f'  joblib_sha256: "{save_stats["sha256"]}"',
    f"  fresh_refit_test_accuracy: {round(fresh_accuracy, 6)}",
    f"  reload_verified_accuracy: {round(reloaded_accuracy, 6)}",
    f"  reload_accuracy_diff: {round(reload_accuracy_diff, 10)}",
    f'  generated_at_utc: "{generated_at}"',
]
write_gate_block(bp1_config_path, persistence_marker, persistence_block_lines)
print(f"[SAVED] {bp1_config_path.relative_to(PROJECT_ROOT)} (model_persistence block)")

# ============================================================
# SECTION 11: Structural integrity checks - raise AssertionError, never silently pass
# ============================================================
checks = {
    "champion_matches_gate3_gate4_gate5_recorded": CHAMPION_NAME == gate3_block["champion_model"] == gate5_summary["champion_model"],
    "fresh_refit_accuracy_matches_gate3_recorded": diff_vs_gate3 < 1e-4,
    "fresh_refit_accuracy_matches_gate5_recorded": diff_vs_gate5 < 1e-6,
    "joblib_file_written": out_path.exists(),
    "joblib_file_nonempty": save_stats["size_bytes"] > 0,
    "reload_round_trip_accuracy_matches_fresh_fit": reload_accuracy_diff < 1e-9,
    "reload_predictions_cover_full_test_set": len(reloaded_pred_labels) == len(X_test),
    "reload_probabilities_sum_to_one": all(
        abs(sum(row) - 1.0) < 1e-6 for row in reload_result["probabilities"][:50]
    ),
    "bundle_has_all_required_keys": {
        "bp_id", "champion_name", "pipeline", "label_encoder", "class_names", "metadata",
    }.issubset(bundle.keys()),
    "metadata_sidecar_written": metadata_path.exists(),
    "model_inventory_updated": inventory_path.exists(),
    "bp1_config_yaml_updated": bp1_config_path.exists(),
}

print("\n=== INTEGRITY CHECKS ===")
for name, passed in checks.items():
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {name}")
    assert passed, f"[CHECK FAILED] {name}"

print(f"\n[ALL CHECKS PASSED] BP1 model persistence complete. Champion ({CHAMPION_NAME}) persisted to "
      f"{out_path.relative_to(PROJECT_ROOT)} ({save_stats['size_bytes']:,} bytes). Reload-verified "
      f"accuracy {round(reloaded_accuracy, 4)} exactly matches the fresh refit and Gate 5's already-"
      f"confirmed {round(gate5_recorded_accuracy, 4)}. Ready for Hardening Step 3 (FastAPI inference service).")
